# 03. Data Engineering + Feature Engineering

**เป้าหมาย**
- ทำความสะอาดข้อมูลให้พร้อมใช้
- สร้าง feature ใหม่ที่มีประโยชน์
- จัดการ missing อย่างมีเหตุผล
- Encode categorical features
- เตรียมข้อมูลสำหรับ Feature Selection และ Modeling

**การตัดสินใจที่ผ่านมา**
- Target = 9 classes (รวม Volcanic activity)
- ทิ้ง: Identifier, high-missing damage, Disaster*, Latitude, Longitude
- เก็บ: Magnitude Scale, Impact features (log + binary), Region, Time features

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 60)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded')

Libraries loaded


## 3.1 โหลดข้อมูล + สร้าง Target + Drop คอลัมน์

In [2]:
df = pd.read_excel('../data/raw/disaster_prediction_dataset.xlsx')

grouping_map = {
    'Flood': 'Flood',
    'Glacial lake outburst flood': 'Flood',
    'Mass movement (wet)': 'Landslide',
    'Mass movement (dry)': 'Landslide',
    'Earthquake': 'Earthquake',
    'Impact': 'Earthquake',
    'Epidemic': 'Epidemic',
    'Infestation': 'Epidemic',
    'Animal incident': 'Epidemic',
    'Extreme temperature': 'Extreme temperature',
    'Fog': 'Extreme temperature',
    'Storm': 'Storm',
    'Drought': 'Drought',
    'Wildfire': 'Wildfire',
    'Volcanic activity': 'Volcanic activity',
}

df['Target'] = df['Disaster Type'].map(grouping_map)

# คอลัมน์ที่ทิ้งทั้งหมด
cols_to_drop = [
    # Identifier
    'DisNo.', 'Classification Key', 'External IDs', 'Event Name',
    # High missing damage
    "Reconstruction Costs ('000 US$)", "Reconstruction Costs, Adjusted ('000 US$)",
    "AID Contribution ('000 US$)", "Insured Damage ('000 US$)", "Insured Damage, Adjusted ('000 US$)",
    # Leakage / Target related
    'Disaster Group', 'Disaster Subgroup', 'Disaster Type', 'Disaster Subtype',
    # Location coordinates (ตัดสินใจทิ้ง)
    'Latitude', 'Longitude',
    # อื่น ๆ ที่อาจไม่จำเป็นหรือ high cardinality มาก
    'Location', 'Admin Units', 'GADM Admin Units', 'River Basin',
    'Entry Date', 'Last Update',
    #Redundant Value 
    'ISO', 'Subregion'
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f'Shape หลัง drop: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Shape หลัง drop: (17756, 25)
Columns: ['Historic', 'Country', 'Region', 'Origin', 'Associated Types', 'OFDA/BHA Response', 'Appeal', 'Declaration', 'Magnitude', 'Magnitude Scale', 'Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day', 'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless', 'Total Affected', "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)", 'CPI', 'Target']


## 3.2 สร้าง Feature ใหม่จากเวลา (Time Features)

In [3]:
# Duration (จำนวนวันโดยประมาณ)
df['duration_days'] = (
    (df['End Year'] - df['Start Year']) * 365 +
    (df['End Month'].fillna(df['Start Month']) - df['Start Month'].fillna(6)) * 30 +
    (df['End Day'].fillna(15) - df['Start Day'].fillna(15))
)
df['duration_days'] = df['duration_days'].clip(lower=0)  # กันค่าติดลบ

# Season จาก Start Month
def get_season(month):
    if pd.isna(month):
        return 'Unknown'
    month = int(month)
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df['Start Month'].apply(get_season)

# Decade
df['decade'] = (df['Start Year'] // 10) * 10

print('Time features created:')
print(df[['Start Year', 'Start Month', 'duration_days', 'season', 'decade']].head(10))
print()
print(df['season'].value_counts())

Time features created:
   Start Year  Start Month  duration_days   season  decade
0        1900          9.0            0.0   Autumn    1900
1        1900          1.0            0.0   Winter    1900
2        1900          1.0            0.0   Winter    1900
3        1900          7.0            0.0   Summer    1900
4        1900          7.0            0.0   Summer    1900
5        1900          NaN            NaN  Unknown    1900
6        1900          NaN            NaN  Unknown    1900
7        1901          NaN            NaN  Unknown    1900
8        1901          8.0            0.0   Summer    1900
9        1902          4.0            0.0   Spring    1900

season
Summer     5114
Autumn     4267
Winter     4240
Spring     3763
Unknown     372
Name: count, dtype: int64


## 3.3 Log Transform + Binary Flags สำหรับ Impact Features

In [4]:
impact_cols = [
    'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
    'Total Affected', "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)"
]

for col in impact_cols:
    if col in df.columns:
        # Binary: มีข้อมูลหรือไม่
        df[f'has_{col}'] = df[col].notna().astype(int)
        
        # Log1p transform (เก็บค่าเดิมไว้ก่อน แล้วแทนที่)
        df[col] = np.log1p(df[col])

print('Impact features หลัง log1p + binary flags:')
print(df[[c for c in df.columns if 'Deaths' in c or 'Affected' in c or 'has_' in c]].head())

Impact features หลัง log1p + binary flags:
   Total Deaths  No. Affected  Total Affected  has_Total Deaths  \
0      8.699681           NaN             NaN                 1   
1      5.707110           NaN             NaN                 1   
2      3.433987           NaN             NaN                 1   
3      3.433987           NaN             NaN                 1   
4      4.948760           NaN             NaN                 1   

   has_No. Injured  has_No. Affected  has_No. Homeless  has_Total Affected  \
0                0                 0                 0                   0   
1                0                 0                 0                   0   
2                0                 0                 0                   0   
3                0                 0                 0                   0   
4                0                 0                 0                   0   

   has_Total Damage ('000 US$)  has_Total Damage, Adjusted ('000 US$)  
0            

## 3.4 จัดการ Missing Values

In [5]:
print('=== Missing ก่อนจัดการ ===')
print(df.isnull().sum().sort_values(ascending=False).head(20))

=== Missing ก่อนจัดการ ===
No. Homeless                         15213
Associated Types                     13488
No. Injured                          13102
Origin                               13039
Magnitude                            12468
Total Damage, Adjusted ('000 US$)    12119
Total Damage ('000 US$)              12041
No. Affected                          7154
Total Deaths                          4979
Total Affected                        4563
Start Day                             3601
End Day                               3491
Magnitude Scale                       2103
End Month                              665
CPI                                    408
Start Month                            372
duration_days                          321
has_Total Deaths                         0
has_No. Injured                          0
decade                                   0
dtype: int64


In [6]:
# Numerical: เติมด้วย median
num_cols_fill = ['Magnitude', 'Start Month', 'Start Day', 'End Month', 'End Day',
                 'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless',
                 'Total Affected', "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)",
                 'CPI', 'duration_days']

for col in num_cols_fill:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

# Categorical: เติมด้วย 'Unknown'
cat_cols_fill = ['Origin', 'Associated Types', 'Magnitude Scale', 'Historic',
                 'OFDA/BHA Response', 'Appeal', 'Declaration', 'ISO', 'Country',
                 'Subregion', 'Region', 'season']

for col in cat_cols_fill:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

print('\n=== Missing หลังจัดการ ===')
print(df.isnull().sum().sum(), 'missing values เหลืออยู่')


=== Missing หลังจัดการ ===
0 missing values เหลืออยู่


## 3.5 Encode Categorical Features

เราจะใช้วิธีผสม:
- **Low cardinality** → One-Hot Encoding
- **High cardinality** (Country, ISO) → Frequency Encoding หรือ Target Encoding ภายหลัง
- ตอนนี้ทำ One-Hot สำหรับตัวที่ cardinality ไม่สูงเกินไปก่อน

In [7]:
# ดู cardinality
cat_features = ['Historic', 'Region', 'Subregion', 'Origin', 'Associated Types',
                'OFDA/BHA Response', 'Appeal', 'Declaration', 'Magnitude Scale', 'season', 'ISO', 'Country']

for col in cat_features:
    if col in df.columns:
        print(f'{col}: {df[col].nunique()} unique')

Historic: 2 unique
Region: 5 unique
Origin: 800 unique
Associated Types: 146 unique
OFDA/BHA Response: 2 unique
Appeal: 2 unique
Declaration: 2 unique
Magnitude Scale: 6 unique
season: 5 unique
Country: 230 unique


In [8]:
# One-Hot สำหรับ low-medium cardinality
onehot_cols = ['Historic', 'Region', 'Origin', 'OFDA/BHA Response', 'Appeal',
               'Declaration', 'Magnitude Scale', 'season']

onehot_cols = [c for c in onehot_cols if c in df.columns]

df_encoded = pd.get_dummies(df, columns=onehot_cols, drop_first=True, dtype=int)

print(f'Shape หลัง One-Hot: {df_encoded.shape}')

Shape หลัง One-Hot: (17756, 843)


In [9]:
# Frequency Encoding สำหรับ high cardinality (Country, Subregion, ISO, Associated Types)
high_card_cols = ['Country', 'Subregion', 'Associated Types']

for col in high_card_cols:
    if col in df_encoded.columns:
        freq = df_encoded[col].value_counts(normalize=True)
        df_encoded[f'{col}_freq'] = df_encoded[col].map(freq)
        df_encoded = df_encoded.drop(columns=[col])

print(f'Shape หลัง Frequency Encoding: {df_encoded.shape}')
print('\nตัวอย่างคอลัมน์สุดท้าย:')
print(df_encoded.columns.tolist()[:30], '...')

Shape หลัง Frequency Encoding: (17756, 843)

ตัวอย่างคอลัมน์สุดท้าย:
['Magnitude', 'Start Year', 'Start Month', 'Start Day', 'End Year', 'End Month', 'End Day', 'Total Deaths', 'No. Injured', 'No. Affected', 'No. Homeless', 'Total Affected', "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)", 'CPI', 'Target', 'duration_days', 'decade', 'has_Total Deaths', 'has_No. Injured', 'has_No. Affected', 'has_No. Homeless', 'has_Total Affected', "has_Total Damage ('000 US$)", "has_Total Damage, Adjusted ('000 US$)", 'Historic_Yes', 'Region_Americas', 'Region_Asia', 'Region_Europe', 'Region_Oceania'] ...


## 3.6 จัดเตรียม X และ y + Train/Test Split

In [10]:
# แยก Target
y = df_encoded['Target']
X = df_encoded.drop(columns=['Target'])

# ลบคอลัมน์ที่ไม่ใช่ตัวเลขที่อาจหลงเหลือ
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print('คอลัมน์ non-numeric ที่จะลบ:', non_numeric)
    X = X.drop(columns=non_numeric)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nTarget distribution:\n{y.value_counts()}')

X shape: (17756, 842)
y shape: (17756,)

Target distribution:
Target
Flood                  6183
Storm                  5053
Earthquake             1651
Epidemic               1622
Landslide               918
Drought                 803
Extreme temperature     730
Wildfire                514
Volcanic activity       282
Name: count, dtype: int64


In [11]:
# Train / Test Split (stratify เพื่อรักษาสัดส่วน class)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test : {X_test.shape[0]:,} samples')
print(f'\nTrain class distribution (%):')
print((y_train.value_counts(normalize=True) * 100).round(2))

Train: 14,204 samples
Test : 3,552 samples

Train class distribution (%):
Target
Flood                  34.82
Storm                  28.46
Earthquake              9.30
Epidemic                9.14
Landslide               5.17
Drought                 4.52
Extreme temperature     4.11
Wildfire                2.89
Volcanic activity       1.59
Name: proportion, dtype: float64


In [12]:
# บันทึกข้อมูลที่พร้อมใช้
X_train.to_csv('X_train_fe.csv', index=False)
X_test.to_csv('X_test_fe.csv', index=False)
y_train.to_csv('y_train_fe.csv', index=False)
y_test.to_csv('y_test_fe.csv', index=False)

print('บันทึกไฟล์เรียบร้อย:')
print('  - X_train_fe.csv')
print('  - X_test_fe.csv')
print('  - y_train_fe.csv')
print('  - y_test_fe.csv')

บันทึกไฟล์เรียบร้อย:
  - X_train_fe.csv
  - X_test_fe.csv
  - y_train_fe.csv
  - y_test_fe.csv


## 3.7 สรุป Feature ที่สร้างและจัดการแล้ว

| ประเภท | Feature ที่ทำ | วิธี |
|--------|---------------|------|
| Time | duration_days, season, decade | สร้างใหม่ |
| Impact | Total Deaths ฯลฯ | log1p + has_* binary |
| Categorical low-card | Region, Magnitude Scale, season... | One-Hot |
| Categorical high-card | Country, Subregion | Frequency Encoding |
| Missing | ทุกคอลัมน์ | median / Unknown |
| Dropped | lat/long, identifier, ISO, damage สูง... | ลบออก | ซํ้ากับ Location 

---

**คำถามก่อนไปขั้นถัดไป:**

1. คุณอยากให้เพิ่ม Feature อะไรอีกไหม? (เช่น interaction, binning)
2. พร้อมไปขั้น **Feature Selection + Baseline Modeling** แล้วหรือยัง?

ตอบมาได้เลยครับ